# Road-Sense — YOLOv11 Training Notebook

Interactive training pipeline for the KITTI object detection dataset.

**What this notebook does:**
1. Verify dataset is ready
2. Load and inspect the model
3. Run training with configurable hyperparameters
4. Visualize results (metrics, confusion, predictions)
5. Export the best model for deployment

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Resolve project root whether notebook is started from repo root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "training.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import yaml
from IPython.display import Image, display

from src.models import YOLOTrainer, get_model_info, list_available_models, load_config, load_model

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Project Root: {PROJECT_ROOT}")

## 1. Verify Dataset

In [ ]:
DATA_YAML = PROJECT_ROOT / "data/processed/kitti/data.yaml"

if not DATA_YAML.exists():
    print("Dataset NOT found! Run preprocessing first:")
    print("  python -m src.data.preprocess_dataset")
else:
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)

    data_root = Path(data_cfg.get("path", PROJECT_ROOT))
    if not data_root.is_absolute():
        data_root = (PROJECT_ROOT / data_root).resolve()

    print("Dataset Configuration:")
    print(f"  Classes: {data_cfg['nc']}")
    print(f"  Names:   {data_cfg['names']}")
    train_dir = (
        (data_root / data_cfg["train"]).resolve()
        if not Path(data_cfg["train"]).is_absolute()
        else Path(data_cfg["train"])
    )
    val_dir = (
        (data_root / data_cfg["val"]).resolve() if not Path(data_cfg["val"]).is_absolute() else Path(data_cfg["val"])
    )
    test_dir = (
        (data_root / data_cfg["test"]).resolve() if not Path(data_cfg["test"]).is_absolute() else Path(data_cfg["test"])
    )
    print(f"  Train:   {train_dir}")
    print(f"  Val:     {val_dir}")
    print(f"  Test:    {test_dir}")

    # Count files
    train_count = len(list(train_dir.glob("*.jpg")))
    val_count = len(list(val_dir.glob("*.jpg")))
    test_count = len(list(test_dir.glob("*.jpg")))
    print("\nImage counts:")
    print(f"  Train: {train_count}")
    print(f"  Val:   {val_count}")
    print(f"  Test:  {test_count}")
    print(f"  Total: {train_count + val_count + test_count}")

## 2. Inspect Model

In [ ]:
# List all available models
models = list_available_models()
print(f"{'Model':<15} {'Params (M)':<12} {'Size (MB)':<12}")
print("-" * 40)
for m in models:
    print(f"{m['name']:<15} {m['params_m']:<12.1f} {m['size_mb']:<12.1f}")

In [ ]:
# Load the model
MODEL_NAME = "yolo11m"  # Change to yolo11s, yolo11l, etc.

model = load_model(MODEL_NAME, pretrained=True)
info = get_model_info(model)

print(f"Model: {MODEL_NAME}")
print(f"Type: {info['model_type']}")
print(f"Parameters: {info['num_parameters'] / 1e6:.2f}M")
print(f"Size: ~{info['size_mb']:.1f} MB")
print(f"Classes ({info['num_classes']}): {list(info['class_names'].values())}")

## 3. Configure Training

In [ ]:
# Load the same default config used by train.py
config = load_config(PROJECT_ROOT / "configs/training.yaml")

# Notebook overrides (mirrors CLI-style overrides in train.py)
NOTEBOOK_OVERRIDES = {
    # "model": "yolo11s",
    # "epochs": 50,
    # "batch_size": 16,
    # "imgsz": 640,
    # "device": "0",
    # "workers": 8,
    # "data": "data/processed/kitti/data.yaml",
    # "name": "exp_notebook",
}

if "model" in NOTEBOOK_OVERRIDES:
    config["model"]["name"] = NOTEBOOK_OVERRIDES["model"]
if "epochs" in NOTEBOOK_OVERRIDES:
    config["training"]["epochs"] = NOTEBOOK_OVERRIDES["epochs"]
if "batch_size" in NOTEBOOK_OVERRIDES:
    config["data"]["batch_size"] = NOTEBOOK_OVERRIDES["batch_size"]
if "imgsz" in NOTEBOOK_OVERRIDES:
    config["data"]["imgsz"] = NOTEBOOK_OVERRIDES["imgsz"]
if "device" in NOTEBOOK_OVERRIDES:
    config["device"]["device"] = NOTEBOOK_OVERRIDES["device"]
if "workers" in NOTEBOOK_OVERRIDES:
    config["data"]["workers"] = NOTEBOOK_OVERRIDES["workers"]
if "data" in NOTEBOOK_OVERRIDES:
    config["data"]["yaml_path"] = NOTEBOOK_OVERRIDES["data"]
if "name" in NOTEBOOK_OVERRIDES:
    config["logging"]["name"] = NOTEBOOK_OVERRIDES["name"]

# Keep notebook behavior aligned with script defaults
config["device"]["half_precision"] = bool(torch.cuda.is_available())
if not torch.cuda.is_available():
    config["device"]["device"] = "cpu"

# Print summary
print("Training Configuration Summary:")
print(f"  Model:      {config['model']['name']}")
print(f"  Epochs:     {config['training']['epochs']}")
print(f"  Batch:      {config['data']['batch_size']}")
print(f"  Image Size: {config['data']['imgsz']}")
print(f"  Device:     {config['device']['device']}")
print(f"  FP16:       {config['device']['half_precision']}")
print(f"  Classes:    {config['data']['yaml_path']}")
print(f"  Save to:    {config['logging']['project']}/{config['logging']['name']}")

## 4. Train

In [ ]:
# Initialize trainer
trainer = YOLOTrainer(
    config=config,
    config_path=str(PROJECT_ROOT / "configs/training.yaml"),
    project_root=str(PROJECT_ROOT),
)

trainer.setup()
print("Setup complete. Starting training...\n")

# Run training
results = trainer.train()

In [ ]:
# Training summary
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Model: {config['model']['name']}")
print(f"Epochs trained: {results['epochs_trained']}")
print(f"Results saved: {results['save_dir']}")

if results["metrics"]:
    print("\nFinal Metrics:")
    for key, value in sorted(results["metrics"].items()):
        print(f"  {key}: {value:.4f}" if isinstance(value, (int, float)) else f"  {key}: {value}")

## 5. Visualize Results

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Plot training curves
results_dir = Path(results["save_dir"]) if results.get("save_dir") else PROJECT_ROOT / "runs/train"

# Check for results.png (training curves)
results_png = results_dir / "results.png"
if results_png.exists():
    display(Image(str(results_png)))
else:
    print(f"Training curves not yet available at: {results_png}")

In [ ]:
# Plot confusion matrix
confusion_matrix = results_dir / "confusion_matrix.png"
if confusion_matrix.exists():
    display(Image(str(confusion_matrix)))
else:
    print(f"Confusion matrix not yet available at: {confusion_matrix}")

In [ ]:
# Plot F1 curve and PR curve
for plot_name in ["F1_curve.png", "PR_curve.png"]:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        display(Image(str(plot_path)))

In [ ]:
# Plot training batch predictions
train_batch_path = results_dir / "train_batch0.jpg"
if train_batch_path.exists():
    display(Image(str(train_batch_path), width=800))
else:
    print(f"Training batch predictions not yet available at: {train_batch_path}")

## 6. Test Predictions on Sample Images

In [ ]:
import random

# Load best trained checkpoint for inference (not the initial pretrained model)
best_ckpt = Path(results["save_dir"]) / "weights" / "best.pt"
infer_model = load_model(model_name=config["model"]["name"], weights_path=str(best_ckpt), pretrained=False)

# Find a sample image from the validation set
val_images = list((PROJECT_ROOT / "data/processed/kitti/images/val").glob("*.jpg"))
sample_image = random.choice(val_images)
print(f"Testing on: {sample_image.name}")

# Run inference
predictions = infer_model.predict(
    source=str(sample_image),
    conf=0.25,
    iou=0.45,
    save=False,
)

# Display result
result = predictions[0]
result_image = result.plot()
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Predictions on {sample_image.name}")
plt.show()

# Print detection details
print(f"\nDetected {len(result.boxes)} objects:")
for box in result.boxes:
    cls_id = int(box.cls[0])
    cls_name = result.names[cls_id]
    conf = float(box.conf[0])
    print(f"  {cls_name}: {conf:.3f}")

In [ ]:
# Batch prediction on multiple validation images
NUM_SAMPLES = 4
sample_images = random.sample(val_images, min(NUM_SAMPLES, len(val_images)))

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    pred = infer_model.predict(str(img_path), conf=0.25, iou=0.45, save=False)
    result_image = pred[0].plot()
    axes[i].imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    axes[i].axis("off")
    axes[i].set_title(f"{img_path.name}")

plt.tight_layout()
plt.show()

## 7. Validate on Test Set

In [ ]:
# Run formal validation
val_metrics = trainer.validate()
print("Test Set Metrics:")
for key, value in sorted(val_metrics.items()):
    print(f"  {key}: {value:.4f}" if isinstance(value, (int, float)) else f"  {key}: {value}")